# Smart Buildings — Submission Version 4 (explained implementation)

**Experiment:** Short-horizon forecasting plus HVAC supervisory decisions. This is a non-PID direction compared with V2 and V3. V4's Kaggle leaderboard score is **not yet known**; the examples below illustrate code behavior, not measured performance.

## 1. Architecture: one five-minute decision

~~~text
             Observation vector (current building)
                            |
                       _schedule
                  comfort / eco / prestart
                            |
                       _read_zone
                  15-minute zone forecasts
                            |
                      _choose_mode
                 AHU OFF / COOL / REHEAT
                            |
       +--------------------+---------------------+
       |                                          |
 _set_fan_pressure / _set_supply_air       _set_zone_actuators
       |                                    each VAV damper + valve
       +--------------------+---------------------+
                            |
                       _set_hot_water
             pump ON only for real reheat demand
                            |
       continuous_action float32[-1,+1] and
          discrete_action int32{0,1}
~~~

**Plant constraints:** AHUs can **cool but cannot directly heat**; heating comes from hot-water-fed VAV reheat coils. Continuous action 0 means **half open**, not OFF. A running AHU has a minimum static pressure, its recirculation is capped at 70%, and VAV dampers have a 10% minimum opening. Device positions are taken from live building metadata, not fixed action offsets.

## 2. Method reference (every method)

| Method | Responsibility | Illustration |
| --- | --- | --- |
| **TransferPolicy.__init__** | Build sensor/actuator lookup tables | room_7 maps to VAV_7; AHUs sorted numerically |
| **reset** | Reset trends and equipment-on history | New episode has no stale state |
| **_clamp** | Clip a value to a range | clamp(1.2) = 1.0 |
| **_norm_fraction** | Convert a physical fraction to normalized action | 0% -> -1, 50% -> 0, 100% -> +1 |
| **_schedule** | Choose comfort, eco, or prestart limits | Weekday 08:45 can prestart |
| **_read_zone** | Read live sensors; EWMA + 15-minute prediction | Predicted 298 K vs 297 K limit = 1 K hot |
| **_choose_mode** | Stateful AHU off/cool/reheat decision | Start 0.22 K; stop 0.12 K |
| **_recirculation** | Calculate free-cooling outdoor air intake | Returns normalized mixing request |
| **_empty_actions** | Initialize correctly typed safe action arrays | Dampers/valves -1; AHU off |
| **_observe_ahu_zones** | Collect valid forecast records for an AHU | Invalid sensors skipped |
| **_set_fan_pressure** | Fan pressure based on severity and zone coverage | Request 200–1750 Pa |
| **_set_supply_air** | AHU cooling setpoint and economizer | Target 285–296 K when cooling |
| **_set_zone_actuators** | Individual damper and reheat valve control | Cold zone gets valve + water demand |
| **_set_hot_water** | Shared pump/water reset from valve requests | 75th-percentile heating demand |
| **get_action** | Orchestrate the full decision per step | Action dict from observation vector |

Nested helpers are also documented: **ahu_sort_key** sorts unit identifiers numerically and **cyclic** decodes time-of-day/day-of-week sine/cosine features.

## 3. Worked examples (hypothetical)

**Cooling:** Forecast zone temperature is 298 K; its upper band edge is 297 K. The forecast excursion is 1 K hot. The controller starts the relevant AHU, lowers its cooling supply setpoint and opens that VAV damper. The reheat valve stays closed.

**Heating:** Forecast temperature is 292.5 K and the lower edge is 294 K. A 1.5 K cold excursion activates the AHU for airflow, but requests a high AHU temperature setpoint so the AHU does not add cooling. The VAV reheat valve opens and the central water system starts.

**Mixed demand:** A shared AHU serves one hot and one cold room. The dominant excursion chooses the AHU's operating mode. Even when the AHU cools, the cold zone may receive reheat through its own VAV, so the water system stays active.

**Preconditioning and hysteresis:** On a weekday at 08:45, the policy can use the upcoming comfort band 30 minutes early. An AHU starts at >=0.22 K forecast excursion, stays on at least three 5-minute steps, and subsequently stops when demand falls to <=0.12 K.

## 4. Experimental knobs and validation

Set **USE_TREND**, **USE_HYSTERESIS**, or **USE_PRECONDITION** to False one at a time for ablations. The thresholds and water/pressure curves are **heuristics**; no empirical performance improvement is claimed. Run the offline action-contract tests, then full **30 public scenarios × 8,640 steps** using the [organizer's starter kit](https://github.com/smart-buildings-2026/starter-kit). Compare comfort, energy, carbon and total loss with V3 (public score 0.06207).

**Submission:** CPU, internet off, no external model; the final cell starts the Kaggle inference gateway, which runs evaluations and produces submission artifacts.


In [ ]:
"""Version 4: demand-driven HVAC supervisory control with short-term prediction.

The controller works from live building metadata; this is NOT an AHU PID
controller or a trained model. All internal temperatures use kelvin.
The inference gateway calls get_action() every five simulated minutes.
"""
import re
import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


class TransferPolicy(smart_buildings_inference_server.Policy):
    """Turn building observations into normalized HVAC commands.

    One shared AHU supplies air to multiple zones, but each VAV box has its
    own damper and hot-water reheat valve. The AHU CAN COOL but cannot heat;
    heating comes exclusively from hot-water VAV reheat coils.

    Pipeline: schedule -> zone forecast -> AHU mode -> fan and zone commands
              -> hot-water demand gate -> final action arrays.

    Persistent state: prev_temp/trend track each zone's recent drift;
    running/run_steps provide AHU start/stop hysteresis.

    A single instance is configured for one building through public metadata.
    """

    # Ablation switches: disable independently to measure their effect.
    USE_TREND = True
    USE_HYSTERESIS = True
    USE_PRECONDITION = True
    PREDICTION_STEPS = 3  # 3 x 5-minute samples: forecast 15 min ahead
    START_K = 0.22    # AHU starts above this forecast excursion (K)
    STOP_K = 0.12     # An AHU may stop below this demand (K)
    MIN_RUN_STEPS = 3  # Minimum run is 15 minutes, to limit cycling
    PRESTART_HOURS = 0.5  # Precondition for 30 min before occupancy

    def __init__(self, building_meta_data, action_spec):
        """Map live device metadata to array positions once, at construction.

        Args:
            building_meta_data: Per-building observation names, actuator
                positions, zone-to-AHU assignments and normalization stats.
            action_spec: Competition action specification, passed to the
                parent Kaggle policy interface.

        Example: room_7 maps to VAV_7; AHU IDs sort numerically so ahu_2
        precedes ahu_10. No building-specific shape is hard-coded.
        """
        super().__init__(building_meta_data, action_spec)
        # All indexes are read from metadata. Continuous slots are interleaved,
        # not grouped into separate temperature/pressure/VAV blocks.
        m = building_meta_data
        self.n_cont = int(m['n_continuous_action'])
        self.n_disc = int(m['n_discrete_action'])
        self.temp_idx = list(map(int, m['temp_indices']))
        self.press_idx = list(map(int, m['pressure_indices']))
        self.recirc_idx = list(map(int, m['recirculation_indices']))
        self.water_idx = list(map(int, m['water_indices']))
        self.n_ahu = len(self.temp_idx)
        self.water_pressure_idx = (self.press_idx[self.n_ahu]
                                   if len(self.press_idx) > self.n_ahu else None)
        self.water_temperature_idx = self.water_idx[0] if self.water_idx else None
        self.water_run_idx = self.n_disc - 1
        self.fields = {name: i for i, name in enumerate(m['observation_field_names'])}
        self.comfort = tuple(map(float, m['comfort_temp_window']))
        self.eco = tuple(map(float, m['eco_temp_window']))
        self.morning = float(m['morning_start_hour'])
        self.evening = float(m['evening_start_hour'])
        norm = m.get('observation_normalization_metadata', {}).get(
            'zone_air_temperature_sensor', {})
        self.zone_mean = float(norm.get('sample_mean', 0.0))
        self.zone_std = float(norm.get('sample_std', 1.0))
        if not np.isfinite(self.zone_std) or self.zone_std <= 0:
            self.zone_std = 1.0
        assignments = m['ahu_zone_assignments']
        action_map = m['zone_action_indices']

        def ahu_sort_key(name):
            """Numeric ordering prevents ahu_10 sorting before ahu_2."""
            match = re.search(r'\d+', str(name))
            return (int(match.group()) if match else -1, str(name))

        self.ahus = []
        self.zone_ids = []
        for i, ahu_id in enumerate(sorted(assignments, key=ahu_sort_key)[:self.n_ahu]):
            zones = []
            for room in assignments[ahu_id]:
                vav = str(room).replace('room_', 'VAV_')
                zone_index = self.fields.get(vav + '_zone_air_temperature_sensor')
                if zone_index is None:
                    continue
                actions = action_map.get(vav, {})
                zones.append({
                    'id': vav,
                    'temp': zone_index,
                    'lo': self.fields.get(vav + '_zone_air_heating_temperature_setpoint_sensor'),
                    'hi': self.fields.get(vav + '_zone_air_cooling_temperature_setpoint_sensor'),
                    'damper': actions.get('supply_air_damper_percentage_command'),
                    'reheat': actions.get('reheat_valve_percentage_command'),
                })
                self.zone_ids.append(vav)
            self.ahus.append({
                'zones': zones,
                'temp': self.temp_idx[i],
                'pressure': self.press_idx[i] if i < len(self.press_idx) else None,
                'recirc': self.recirc_idx[i] if i < len(self.recirc_idx) else None,
                'outside': self.fields.get(f'ahs_{ahu_id}_outside_air_temperature_sensor'),
                'run': i,
            })
        self.reset()

    def reset(self):
        """Clear episode history but keep the building's metadata index maps.

        Called by the gateway for a fresh simulation; prevents stale
        temperatures, slopes or run-duration state leaking across episodes.
        """
        self.prev_temp = {}
        self.trend = {}
        self.running = [False] * len(self.ahus)
        self.run_steps = [0] * len(self.ahus)

    @staticmethod
    def _clamp(x, lo=0.0, hi=1.0):
        """Clip any scalar to the inclusive interval [lo, hi].

        Example: _clamp(1.3) returns 1.0; with lo=-0.6, hi=0.6 it
        instead accepts both negative and positive temperature trends.
        """
        return float(np.clip(x, lo, hi))

    @staticmethod
    def _norm_fraction(x):
        """Translate a physical 0..1 fraction into a normalized -1..+1 action.

        Examples: 0 percent -> -1, 50 percent -> 0, 100 percent -> +1.
        The actual HVAC plant additionally caps maximum recirculation to 70%.
        """
        return 2.0 * float(np.clip(x, 0.0, 1.0)) - 1.0

    def _schedule(self, obs):
        """Decode schedule sensors; choose a temperature band and prestart flag.

        Returns:
            (band, prestart), where band=(minimum_K, maximum_K).
            Weekday working hours use the narrower comfort band, other
            times the eco band. An optional 30-minute prestart uses the
            upcoming comfort band before occupied hours begin.

        If cyclic features are unavailable, fall back to the eco band.
        """
        def cyclic(s, c, period):
            """Decode a sine/cosine clock to time within the given period."""
            a, b = self.fields.get(s), self.fields.get(c)
            if a is None or b is None:
                return None
            x, y = float(obs[a]), float(obs[b])
            if not (np.isfinite(x) and np.isfinite(y)):
                return None
            return ((np.arctan2(x, y) % (2.0 * np.pi)) * period / (2.0 * np.pi))
        hour = cyclic('hod_sin_000', 'hod_cos_000', 24.0)
        day = cyclic('dow_sin_000', 'dow_cos_000', 7.0)
        if hour is None or day is None:
            return self.eco, False
        weekday = int(np.floor(day + 1e-6)) < 5
        occupied = weekday and self.morning <= hour < self.evening
        prestart = (self.USE_PRECONDITION and weekday and
                    self.morning - self.PRESTART_HOURS <= hour < self.morning)
        return self.comfort if occupied or prestart else self.eco, prestart

    def _read_zone(self, obs, zone, fallback):
        """Read zone sensors, forecast temperature and compute comfort excursions.

        1. Denormalize zone air temperature to kelvin.
        2. Prefer valid current low/high setpoint sensors; otherwise use
           the schedule fallback (or the upcoming band for prestart).
        3. Smooth each 5-minute temperature change with an EWMA
           (70% previous trend, 30% latest change; max +/-0.30 K).
        4. Forecast 3 steps ahead, bounding the total lead to +/-0.60 K.
        5. Return hot/cold distance outside the predicted comfort band.

        Example: predicted=298 K, upper limit=297 K -> hot=1 K, cold=0.
        Returns None for an invalid zone temperature; otherwise a dict.
        Updates per-zone history as a side effect.
        """
        t = self.zone_mean + float(obs[zone['temp']]) * self.zone_std
        if not np.isfinite(t) or t < 240 or t > 340:
            return None
        low = float(obs[zone['lo']]) if zone['lo'] is not None else float('nan')
        high = float(obs[zone['hi']]) if zone['hi'] is not None else float('nan')
        if not (np.isfinite(low) and np.isfinite(high) and
                245 <= low < high <= 335 and high - low <= 35):
            low, high = fallback
        # The sensor represents the current scoring band. Only the intentional
        # 30-minute prestart uses the forthcoming comfort band instead.
        if self._prestart:
            low, high = fallback
        key = zone['id']
        previous = self.prev_temp.get(key)
        if previous is not None:
            delta = self._clamp(t - previous, -0.30, 0.30)
            self.trend[key] = 0.70 * self.trend.get(key, 0.0) + 0.30 * delta
        else:
            self.trend[key] = 0.0
        self.prev_temp[key] = t
        slope = self.trend[key] if self.USE_TREND else 0.0
        predicted = t + self.PREDICTION_STEPS * slope
        # Protect against deadband drift: do not lead more than 0.6 K.
        predicted = t + self._clamp(predicted - t, -0.6, 0.6)
        hot = max(0.0, predicted - high)
        cold = max(0.0, low - predicted)
        return {'zone': zone, 'temp': t, 'hot': hot, 'cold': cold,
                'low': low, 'high': high}

    def _choose_mode(self, i, hot, cold):
        """Choose 'off', 'cool' or 'heat' for one AHU with hysteresis.

        Inputs: AHU index and largest predicted heating/cooling excursions
        among the AHU's zones, in kelvin. Return is the operating mode.

        Start above 0.22 K, keep running for at least three 5-minute steps,
        and stop at <=0.12 K after that. 'Heat' means VAV hot-water reheat
        with airflow; AHUs themselves cannot heat.
        The per-AHU state and run-step counter are updated.
        """
        demand = max(hot, cold)
        if not self.USE_HYSTERESIS:
            on = demand >= self.START_K
        elif not self.running[i]:
            on = demand >= self.START_K
        elif self.run_steps[i] < self.MIN_RUN_STEPS:
            on = True
        else:
            on = demand > self.STOP_K
        self.running[i] = bool(on)
        self.run_steps[i] = (self.run_steps[i] + 1) if on else 0
        if not on:
            return 'off'
        return 'cool' if hot >= cold and hot >= 0.08 else 'heat'

    def _recirculation(self, outdoor, indoor, target, pressure_pa):
        """Compute normalized recirculation for economically favorable free cooling.

        Extra outdoor air is worthwhile only when indoor-outdoor delta
        exceeds estimated incremental fan cost: pressure_pa*0.00322 K/Pa.
        A mixing balance sets the outside-air fraction (minimum 30%).
        Returns +1 when maximum physical recirculation is preferred.
        """
        # Full recirculation command is physically clamped to 70% by simulator.
        if not np.isfinite(outdoor) or outdoor >= indoor:
            return 1.0
        if indoor - outdoor <= pressure_pa * 0.00322:
            return 1.0
        # Do not overdraw cold outside air: min intake fraction is 30%.
        fraction_outdoor = self._clamp((indoor - target) /
                                       max(1e-6, indoor - outdoor), 0.30, 1.0)
        return self._norm_fraction(1.0 - fraction_outdoor)

    def _empty_actions(self):
        """Prepare correctly typed action arrays at safe/resting commands.

        A value of zero is a 50% physical setting, not OFF. VAV dampers and
        valves default to -1 (minimum), AHU cooling setpoints to +1 (warmest
        request), recirculation to +1 (plant-clamped at 70%), and run flags
        to 0. Returns (float32 continuous, int32 discrete) action arrays.
        """
        continuous = np.full(self.n_cont, -1.0, dtype=np.float32)
        discrete = np.zeros(self.n_disc, dtype=np.int32)
        for index in self.temp_idx:
            continuous[index] = 1.0
        for index in self.recirc_idx:
            continuous[index] = 1.0
        return continuous, discrete

    def _observe_ahu_zones(self, obs, ahu, fallback_band):
        """Collect validated, predicted zone records for one AHU.

        Each returned dict describes a VAV and its hot/cold demand in kelvin.
        Discard a zone when its temperature sensor is invalid; return [] if
        no zone can be trusted, so the affected AHU can safely idle.
        """
        zone_records = []
        for zone in ahu['zones']:
            record = self._read_zone(obs, zone, fallback_band)
            if record is not None:
                zone_records.append(record)
        return zone_records

    def _set_fan_pressure(self, ahu, zone_records, hot, cold, continuous):
        """Set supply fan pressure from the worst excursion and zone coverage.

        The target is 200..1750 Pa: 200 + 1550 * severity/3 K *
        sqrt(fraction-of-zones-demanding-air), with clipping and a 30%
        minimum coverage factor. This avoids full fan pressure for a single
        minor excursion. Returns the physical pressure, needed by economizer.
        """
        severity_k = max(hot, cold)
        n_demanding = sum(
            max(record['hot'], record['cold']) >= 0.15
            for record in zone_records
        )
        coverage = max(0.30, n_demanding / max(1, len(zone_records)))
        pressure_pa = 200.0 + 1550.0 * self._clamp(severity_k / 3.0) * np.sqrt(coverage)
        pressure_pa = float(np.clip(pressure_pa, 200.0, 1750.0))

        if ahu['pressure'] is not None:
            continuous[ahu['pressure']] = self._clamp(
                pressure_pa / 1000.0 - 1.0, -1.0, 1.0)
        return pressure_pa

    def _set_supply_air(self, obs, ahu, zone_records, mode, hot, pressure_pa, continuous):
        """Set AHU cooling target and outdoor-air economizer mixing.

        COOL: target supply = clip(293 - 2.75*hot, 285, 296) kelvin;
        map 285..305 K to a normalized command, and economize when useful.
        HEAT: set AHU cooling target to +1 (305 K, no active heating).
        Actual heating is controlled by each VAV reheat coil, not the AHU.
        """
        if mode == 'cool':
            target = np.clip(293.0 - 2.75 * hot, 285.0, 296.0)
            continuous[ahu['temp']] = self._clamp(
                (target - 285.0) / 10.0 - 1.0, -1.0, 1.0)

            outside_index = ahu['outside']
            outdoor = (float(obs[outside_index]) if outside_index is not None
                       else float('nan'))
            if ahu['recirc'] is not None:
                indoor = float(np.mean([record['temp'] for record in zone_records]))
                continuous[ahu['recirc']] = self._recirculation(
                    outdoor, indoor, float(target), pressure_pa)
        else:
            continuous[ahu['temp']] = 1.0

    def _set_zone_actuators(self, zone_records, mode, continuous, heating_demands):
        """Give each VAV its own damper and optional hot-water reheat command.

        COOL: hot zones receive proportional airflow; a cold neighbor can
        still receive some airflow and reheat while the shared AHU is cooling.
        HEAT: airflow tracks the zone's cold excursion.

        Append a heating request only when the zone's reheat valve exists and
        actually opens. The hot-water supervisor uses these requests below.
        """
        for record in zone_records:
            zone = record['zone']
            if mode == 'cool':
                airflow = max(self._clamp(record['hot'] / 2.5),
                              0.30 * self._clamp(record['cold'] / 1.5))
            else:
                airflow = self._clamp(record['cold'] / 2.5)

            if zone['damper'] is not None:
                continuous[zone['damper']] = self._norm_fraction(airflow)

            if record['cold'] > 0.08 and zone['reheat'] is not None:
                valve_fraction = self._clamp(record['cold'] / 2.5)
                continuous[zone['reheat']] = self._norm_fraction(valve_fraction)
                heating_demands.append(record['cold'])

    def _set_hot_water(self, heating_demands, continuous, discrete):
        """Run the central water system only when VAV reheat is requested.

        The 75th-percentile cold excursion is the robust building-wide
        heating signal. It scales water setpoint 318..348 K (mapped from the
        native 310..350 K range) and pump pressure with a 45% demand floor.
        No heating request -> leave the water system off.
        """
        if not heating_demands:
            return

        representative_shortfall = float(np.percentile(heating_demands, 75))
        demand_fraction = self._clamp(representative_shortfall / 3.0)
        if self.water_temperature_idx is not None:
            water_temperature_k = 318.0 + 30.0 * demand_fraction
            continuous[self.water_temperature_idx] = self._norm_fraction(
                (water_temperature_k - 310.0) / 40.0)
        if self.water_pressure_idx is not None:
            continuous[self.water_pressure_idx] = self._norm_fraction(
                max(0.45, demand_fraction))
        if 0 <= self.water_run_idx < self.n_disc:
            discrete[self.water_run_idx] = 1

    def get_action(self, obs):
        """Compute one five-minute HVAC action from current observations.

        1. Schedule: select comfort/eco targets or a prestart band.
        2. Forecast zone temperatures using recent per-zone history.
        3. Choose each AHU's operating mode with hysteresis.
        4. Set supply, fan, and per-zone actuator commands for active AHUs.
        5. Gate the shared hot-water system on actual reheat requests.
        6. Sanitize and return the normalized, correctly typed arrays.

        Args:
            obs: Current flat observation vector for this building.
        Returns:
            dict with continuous_action (float32 in [-1,+1]) and
            discrete_action (int32 zeros/ones), both sized by metadata.
        """
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        fallback_band, self._prestart = self._schedule(obs)
        continuous, discrete = self._empty_actions()
        heating_demands = []

        for ahu_index, ahu in enumerate(self.ahus):
            zones = self._observe_ahu_zones(obs, ahu, fallback_band)
            if not zones:
                self.running[ahu_index] = False
                self.run_steps[ahu_index] = 0
                continue

            hottest = max(record['hot'] for record in zones)
            coldest = max(record['cold'] for record in zones)
            mode = self._choose_mode(ahu_index, hottest, coldest)
            if mode == 'off':
                continue

            discrete[ahu['run']] = 1
            pressure_pa = self._set_fan_pressure(
                ahu, zones, hottest, coldest, continuous)
            self._set_supply_air(
                obs, ahu, zones, mode, hottest, pressure_pa, continuous)
            self._set_zone_actuators(zones, mode, continuous, heating_demands)

        self._set_hot_water(heating_demands, continuous, discrete)
        np.nan_to_num(continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(continuous, -1.0, 1.0, out=continuous)
        return {'continuous_action': continuous, 'discrete_action': discrete}


In [ ]:
# Start the official Kaggle inference server; it executes scored episodes.
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
